# Place cell identification from calcium imaging data

Tested on Python 3.9.18 / 3.10.8, with Caiman 1.10.4

Revised from the paper 'A probabilistic framework for decoding behavior from calcium imaging data'. 

In [ ]:
import os
import logging
from __future__ import annotations
import numpy as np
import pandas as pd
from scipy.interpolate import interp1d
import re
import random

import seaborn as sns
import pynapple as nap
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter1d
import plotly.graph_objects as go
from tqdm import tqdm
from joblib import Parallel, delayed
from scipy.ndimage import label
from pathlib import Path
from joblib import Parallel, delayed
from tqdm import tqdm

source = Path(r"F:\Data\VR\A30\20260327")

add_frames = True
slice_no = 1   # starts from 1
slice_size = 10000

# Load the data

In [ ]:

behavior_path = source / "outputs"/ "behavior.nwb" #行为学数据.nwb

output_path = source / "outputs"
os.makedirs(output_path, exist_ok=True)

# 设置日志配置
if slice_no == 1:
    log_filename = output_path / f"output_{slice_no}.log"
else: log_filename = None

logging.basicConfig(
    filename=output_path / f"output_{slice_no}.log",  # 日志文件名
    level=logging.INFO,  
    format='%(asctime)s - %(levelname)s - %(message)s' 
)

def print_to_log(*args, **kwargs):
    print(*args, **kwargs)  # 控制台输出
    logging.info(" ".join(map(str, args)))

track_length = 400
frame_rate = 30
bin_num = 80

behavior_data = nap.load_file(behavior_path)

print_to_log(behavior_data)


In [ ]:
position = behavior_data['SpatialSeries']
epochs = behavior_data['Running_laps']
lick = behavior_data['licking']

position_values = np.array(position) 

scaled_values = position_values * 1  # 位置值从 0-200 → 0-400

# 创建新的 Tsd 对象（时间戳完全不变）
scaled_position = nap.Tsd(
    t=position.times(),  # 原始时间戳不变
    d=scaled_values,     # 缩放后的 NumPy 数组
    time_units='s'
)
position = scaled_position

# np.where(transients==np.max(transients))

print_to_log(position)
# print_to_log(transients)
print_to_log(epochs)

plt.figure(figsize=(10,2))
# plt.plot(transients[1:36000,200], linewidth=0.5)
plt.show()

# print(transients)


In [ ]:
print(df.head())
print(df.shape)

rep = df.iloc[:, 4].to_numpy()
print("rep unique:", np.unique(rep)[:50])
print("rep min/max:", np.min(rep), np.max(rep))

In [ ]:
pos_nav = position.restrict(
    nap.IntervalSet(
        start=epochs.start[1:],
        end=epochs.end[1:]
    )
)

In [ ]:
epochs

# Interpolate behavioral data

In order to compare neuronal activity to behavior, it would be useful to work with vector of the same size. Fortunately, we have time vectors corresponding to both neuronal activity and behavior so we can perform linear interpolation:

We have now interpolated the position of the mouse to the sampling frequency of our imaging data. Note that there is virtually no difference compared to the orginial signal. If you use a different interpolation method (eg spline or bicubic), it is important to verify it does not introduce some artefacts that could result in erroneous results later on.

# Speed filter

we will identify periods of running (speed > 5 cm.s-1) and focus our analyses on these epochs:

First, we will calculate the velocity

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pynapple as nap

scale = 1.0   # temporal scaling for behavioral data

# ==============================
# 1) 缩放位置和舔舐时序
# ==============================
position = nap.Tsd(
    t=position.times() * scale,
    d=position.values,
    time_units="s"
)

lick = nap.Tsd(
    t=lick.times() * scale,
    d=lick.values,
    time_units="s"
)

# ==============================
# 2) 缩放 epochs（IntervalSet）
# ==============================
_orig = behavior_data["Running_laps"]
epochs = nap.IntervalSet(
    start=_orig.start * scale,
    end=_orig.end * scale
)

# ==============================
# 3) Filtered_Position（不再额外插值）
# ==============================
Filtered_Position = position

# ==============================
# 4) restrict + 拼接：构造 navigation-time Position
# ==============================
t_nav = []
pos_nav = []
lap_lengths = []   # 每一圈拼接后的长度，用于后面标记 lap boundary
t_offset = 0.0

if len(position) == 0:
    raise ValueError("position is empty.")
if len(epochs) == 0:
    raise ValueError("epochs is empty.")

print("position start/end:", position.times()[0], position.times()[-1])
print("epochs start/end:", epochs.start[0], epochs.end[-1])
print("number of laps:", len(epochs))

for i in range(len(epochs)):
    ep_i = nap.IntervalSet(
        start=[epochs.start[i]],
        end=[epochs.end[i]]
    )

    seg = Filtered_Position.restrict(ep_i)
    print(f"lap {i}: start={epochs.start[i]:.3f}, end={epochs.end[i]:.3f}, seg_len={len(seg)}")

    if len(seg) == 0:
        continue

    t = seg.times()
    x = seg.values

    if len(t) < 2:
        continue

    t_nav.append(t - t[0] + t_offset)
    pos_nav.append(x)
    lap_lengths.append(len(seg))

    t_offset += (t[-1] - t[0])

if len(t_nav) == 0:
    raise ValueError(
        "t_nav is empty. No valid navigation segments found. "
        "Please check whether position timestamps overlap with Running_laps."
    )

t_nav = np.concatenate(t_nav)
pos_nav = np.concatenate(pos_nav)

pos_nav_tsd = nap.Tsd(
    t=t_nav,
    d=pos_nav,
    time_units="s"
)

# ==============================
# 5) position bins（仅在 navigation data 上）
# ==============================
position_bins = np.linspace(
    pos_nav.min(),
    pos_nav.max(),
    bin_num + 1
)

bin_indices = np.digitize(pos_nav, position_bins, right=True) - 1
bin_indices = np.clip(bin_indices, 0, bin_num - 1)
bin_indices = pd.Series(bin_indices)

# ==============================
# 6) velocity（navigation-time，自然无 black screen）
# ==============================
dt = np.diff(t_nav)
dx = np.diff(pos_nav)

velocity = np.zeros_like(pos_nav, dtype=float)
valid = dt > 0
velocity[1:][valid] = np.abs(dx[valid] / dt[valid])

# lap 边界速度设为 NaN（语义更干净）
lap_boundary = np.zeros_like(velocity, dtype=bool)
cursor = 0
for seg_len in lap_lengths:
    if cursor > 0 and cursor < len(lap_boundary):
        lap_boundary[cursor] = True
    cursor += seg_len

velocity[lap_boundary] = np.nan

# ==============================
# 7) velocity smoothing（仅对 navigation data）
# ==============================
smoothWindowSize = 29

v = np.nan_to_num(velocity, nan=0.0)
kernel = np.ones(smoothWindowSize, dtype=float)

v_smooth = np.convolve(v, kernel, mode="same")
w = np.convolve((~np.isnan(velocity)).astype(float), kernel, mode="same")

velocity_smooth = np.full_like(v_smooth, np.nan, dtype=float)
mask = w > 0
velocity_smooth[mask] = v_smooth[mask] / w[mask]

velocity = velocity_smooth

# ==============================
# 8) running 判定
# ==============================
min_speed_threshold = 5  # cm/s
is_running = velocity > min_speed_threshold

# ==============================
# 9) 三 Y 轴绘图（navigation time）
# ==============================
fig, ax1 = plt.subplots(figsize=(15, 6))

# --- Position ---
ax1.plot(
    t_nav,
    pos_nav,
    color="green",
    linewidth=2,
    label="Location on track"
)
ax1.set_xlabel("Navigation time (s)", fontsize=16)
ax1.set_ylabel("Location on track", fontsize=16, color="green")
ax1.tick_params(axis="y", labelcolor="green")
ax1.grid(True, linestyle="--", alpha=0.6)

# --- Velocity ---
ax2 = ax1.twinx()
ax2.plot(
    t_nav,
    velocity,
    color="firebrick",
    linewidth=0.6,
    label="Velocity"
)
ax2.set_ylabel("Velocity (cm/s)", fontsize=16, color="firebrick")
ax2.tick_params(axis="y", labelcolor="firebrick")
ax2.set_ylim(0, 100)

# --- Running periods ---
ax3 = ax1.twinx()
ax3.spines["right"].set_position(("outward", 100))

fill_values = np.zeros_like(t_nav, dtype=float)
fill_values[is_running] = track_length

ax3.fill_between(
    t_nav,
    fill_values,
    0,
    facecolor=(0/255, 100/255, 200/255, 0.25),
    label="Running period",
    zorder=0
)
ax3.set_ylim(0, track_length)
ax3.set_yticks([])

# --- Legend ---
h1, l1 = ax1.get_legend_handles_labels()
h2, l2 = ax2.get_legend_handles_labels()
h3, l3 = ax3.get_legend_handles_labels()

fig.legend(
    handles=h1 + h2 + h3,
    labels=l1 + l2 + l3,
    loc="upper center",
    bbox_to_anchor=(0.5, 1.0),
    ncol=3,
    fontsize=16
)

fig.tight_layout(rect=[0, 0.05, 1, 0.9])

if slice_no == 1:
    fig.savefig(output_path / "positions_and_velocity_navtime.png", dpi=300)

plt.show()

# ==============================
# 10) lick × trial（restrict + 原始 lap index）
# ==============================
fig, ax = plt.subplots(figsize=(8, 6))

for n in range(len(epochs) - 1, -1, -1):
    ep_n = nap.IntervalSet(
        start=[epochs.start[n]],
        end=[epochs.end[n]]
    )

    lick_seg = lick.restrict(ep_n)
    pos_seg = position.restrict(ep_n)

    if len(lick_seg) == 0 or len(pos_seg) == 0:
        continue

    if len(pos_seg) < 2:
        continue

    lick_positions = np.interp(
        lick_seg.times(),
        pos_seg.times(),
        pos_seg.values
    )

    ax.plot(
        lick_positions,
        np.full_like(lick_positions, n),
        ".",
        markersize=5,
        color="#6FA3C6",
        alpha=0.3
    )

ax.invert_yaxis()
ax.set_xlabel("Position")
ax.set_ylabel("Trial")

if slice_no == 1:
    fig.savefig(output_path / "lick_vs_trial.png", dpi=300)

plt.show()

In [ ]:
t_nav = []
pos_nav = []
t_offset = 0.0

print("position range:", position.times()[0], position.times()[-1])
print("n_epochs:", len(epochs))
print("epoch starts:", epochs.start[:5])
print("epoch ends:", epochs.end[:5])

for i in range(len(epochs)):
    ep_i = nap.IntervalSet(
        start=[epochs.start[i]],
        end=[epochs.end[i]]
    )

    seg = Filtered_Position.restrict(ep_i)
    print(f"lap {i}: start={epochs.start[i]:.3f}, end={epochs.end[i]:.3f}, seg_len={len(seg)}")

    if len(seg) == 0:
        continue

    t = seg.times()
    x = seg.values

    t_nav.append(t - t[0] + t_offset)
    pos_nav.append(x)

    t_offset += t[-1] - t[0]

if len(t_nav) == 0:
    raise ValueError(
        "No valid navigation segments found. "
        "Check whether position timestamps overlap with Running_laps epochs."
    )

t_nav = np.concatenate(t_nav)
pos_nav = np.concatenate(pos_nav)

### Licking selectivity

In [ ]:
lick_pos = position.interpolate(lick)

reward_location = 380
unreward_location = reward_location - 200
area_size = 10

pre_reward = (lick_pos.values >= (reward_location-area_size)) & (lick_pos.values < reward_location)
unrewarded = (lick_pos.values >= (unreward_location)) & (lick_pos.values < (unreward_location + area_size))

num_pre = np.sum(pre_reward)
num_un = np.sum(unrewarded)

selectivity = (num_pre - num_un) / (num_pre + num_un) if num_pre + num_un > 0 else np.nan

print(f"Licking selectivity:{selectivity:.3f}")

In [ ]:
# ============================================================
# 构造 t_lick_nav（lick 的 navigation time）
# 必须在使用 t_lick_nav 之前运行
# ============================================================

t_lick_nav_list = []

t_offset = 0.0

for i in range(len(epochs)):
    pos_seg = position.restrict(epochs[i])
    lick_seg = lick.restrict(epochs[i])

    if len(pos_seg) == 0:
        continue

    t0 = pos_seg.times()[0]

    if len(lick_seg) > 0:
        t_lick_nav_list.append(
            lick_seg.times() - t0 + t_offset
        )

    # 累积 navigation time（与 t_nav 构造一致）
    t_offset += pos_seg.times()[-1] - t0

# 拼接为 ndarray
if len(t_lick_nav_list) > 0:
    t_lick_nav = np.concatenate(t_lick_nav_list)
else:
    t_lick_nav = np.array([])


In [ ]:
# ==============================
# 固定时间窗口：0–5 分钟（navigation time）
# ==============================
window_start = 6 * 60         # seconds (navigation time)
window_end   = 11 * 60       # 5 minutes navigation time

# ------------------------------
# window mask（navigation time）
# ------------------------------
win_mask = (t_nav >= window_start) & (t_nav <= window_end)

# ==============================
# Position（navigation only）
# ==============================
t_pos_win = t_nav[win_mask]
pos_win   = pos_nav[win_mask]

pos_plot = nap.Tsd(
    t=t_pos_win,
    d=pos_win,
    time_units="s"
)

# ==============================
# Velocity（navigation only）
# ==============================
vel_win = velocity[win_mask]
t_speed = t_nav[win_mask]

# ==============================
# Lick（navigation only, absolute navigation time）
# ==============================
lick_win_mask = (t_lick_nav >= window_start) & (t_lick_nav <= window_end)
t_lick_win_abs = t_lick_nav[lick_win_mask]   # 绝对 navigation time

# ==============================
# Reward onset（navigation time，基于当前 position）
# ==============================
reward_location = 380  # cm
reward_times = []

for i in range(1, len(pos_win)):
    if pos_win[i - 1] < reward_location and pos_win[i] >= reward_location:
        reward_times.append(t_pos_win[i])

# 每一圈只保留一个 reward onset
reward_times = np.unique(np.array(reward_times))

# ==============================
# 构造相对时间轴（从 window_start = 0）
# ==============================
t_rel_pos   = t_pos_win   - window_start
t_rel_speed = t_speed     - window_start
reward_times_rel = reward_times - window_start

# ============================================================
# 🔑 关键修复：将 lick 映射到 position 的时间轴（防止错位）
# ============================================================

# position 的绝对 navigation time（用于对齐）
t_pos_abs = pos_plot.times()   # == t_pos_win

# 对每个 lick，找到最近的 position 时间点
lick_idx = np.searchsorted(t_pos_abs, t_lick_win_abs, side="left")

# 防止越界
valid = (lick_idx > 0) & (lick_idx < len(t_rel_pos))
lick_idx = lick_idx[valid]

# lick 在当前图上的 x 坐标（相对时间）
t_lick_plot = t_rel_pos[lick_idx]

# ==============================
# 输出变量说明（供后续绘图使用）
# ==============================
# t_rel_pos        : position x 轴
# pos_plot.values  : position y
# t_lick_plot      : lick x 轴（已对齐）
# vel_win          : speed y
# t_rel_speed      : speed x 轴
# reward_times_rel : reward 竖线 x


In [ ]:
# ======================
# 构造相对时间轴（从 0 开始，navigation time）
# ======================
t_rel_pos   = t_pos_win - window_start
t_rel_speed = t_speed   - window_start
speed_plot  = vel_win   # 语义更清晰，后面直接用

reward_location = 380  # cm

pos_vals = pos_plot.values
t_vals   = t_rel_pos   # 注意：就是你画 position 用的横轴

reward_times_rel_plot = []
for i in range(1, len(pos_vals)):
    if pos_vals[i-1] < reward_location and pos_vals[i] >= reward_location:
        reward_times_rel_plot.append(t_vals[i])

reward_times_rel_plot = np.unique(np.array(reward_times_rel_plot))

# t_rel_lick 已在前面构造，这里不再重复
# reward_times_rel 已在前面构造

# ======================
# Figure & axes
# ======================
fig, axes = plt.subplots(
    nrows=3,
    ncols=1,
    figsize=(30, 6),
    sharex=True,
    gridspec_kw={'height_ratios': [1, 0.5, 1.0], 'hspace': 0.25}
)

ax_pos, ax_lick, ax_speed = axes

# ======================
# 1. Position (navigation only)
# ======================
ax_pos.plot(
    t_rel_pos,
    pos_plot.values,
    color='#5B6471',
    linewidth=1.8
)
ax_pos.set_ylabel('Position (cm)')
ax_pos.set_ylim(0, track_length)
ax_pos.spines['top'].set_visible(False)
ax_pos.spines['right'].set_visible(False)

# ======================
# 2. Lick raster (navigation only)
# ======================
ax_lick.vlines(
    t_lick_plot,
    ymin=0,
    ymax=1,
    color="#555453",
    linewidth=0.5
)
ax_lick.set_ylim(0, 1)
ax_lick.set_yticks([0, 1])
ax_lick.set_yticklabels(['0', '1'])
ax_lick.set_ylabel('Lick')
ax_lick.spines['top'].set_visible(False)
ax_lick.spines['right'].set_visible(False)

# ======================
# 3. Speed (navigation only)
# ======================
ax_speed.plot(
    t_rel_speed,
    speed_plot,
    color='#5B6471',
    linewidth=1
)
ax_speed.set_ylabel('Speed (cm/s)')
ax_speed.set_xlabel('Time (s)')
ax_speed.set_ylim(0, 60)
ax_speed.spines['top'].set_visible(False)
ax_speed.spines['right'].set_visible(False)

# ======================
# Reward onset (贯通三行，虚线，navigation time)
# ======================
for rt in reward_times_rel_plot:
    for ax in axes:
        ax.axvline(
            x=rt,
            color="#E2A528",
            linewidth=1.6,
            linestyle="--",
            alpha=0.85,
            zorder=10
        )

# ======================
# X 轴刻度（每 50 s）
# ======================
max_t = t_rel_speed.max()
ax_speed.set_xticks(np.arange(0, max_t + 1, 50))

# ======================
# 全局美化
# ======================
for ax in axes:
    ax.tick_params(axis='both', labelsize=10)
    ax.grid(False)
    ax.tick_params(
        axis='both',
        which='both',
        direction='out',
        length=8,
        width=2,
        bottom=False,
        left=True,
        top=False,
        right=False
    )
ax_pos.spines['bottom'].set_visible(False)

# Lick
ax_lick.spines['bottom'].set_visible(False)
plt.tight_layout()

# ======================
# 保存
# ======================
if slice_no == 1:
    fig.savefig(
        output_path / f"behavior_trace_{window_end/60:.0f}min_navtime.svg",
        dpi=300,
        bbox_inches='tight'
    )

plt.show()


In [ ]:
behavior_analysis_data = {"lick selectivity" : float(selectivity),
                          "velocity" : float(velocity.mean()) }

In [ ]:
df = pd.DataFrame([behavior_analysis_data])
df.to_csv(os.path.join(output_path, "behavior_analysis_data.csv"), index=False)

In [ ]:
velocity.mean()

In [ ]:
behavior_analysis_data